# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HASIB-ALI-RISHAD/ML-Engineering-Internship-FlyRank-AI/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am pursuing **Lane 2: Refresh / Content Opportunity Scoring**.

The starter dataset contains 30,000 content items across 32 pseudonymized clients. A quick look at the data reveals that 16,262 of those pages — over half the inventory — currently show a declining trend. No editorial team has the capacity to manually review thousands of candidates per cycle, which is precisely the problem a scored, ranked queue is designed to solve. A second signal reinforces the lane choice: 25.7% of rows are missing `word_count`, meaning any simple fixed threshold rule (e.g., "flag all thin pages") will silently fail on a quarter of the inventory. A learned scoring model can encode missingness explicitly, rank pages on the signals that *are* present, and produce a reason-coded output that a human reviewer can inspect and trust. Lane 2 is the right fit because the scale of the problem, the incompleteness of the metadata, and the decision-support framing all point to the same need: a principled, transparent ranking — not a rule.

In [1]:
# Supporting numbers for Section 1
import pandas as pd
import os

# The kernel working directory when run via nbconvert from repo root is the repo root itself
# Use os.path for a robust cross-environment path
csv_path = os.path.join(os.getcwd(), 'data', 'raw', 'content_refresh_anonymized.csv')
if not os.path.exists(csv_path):
    # Fallback: two levels up from work/notebooks/
    csv_path = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath('.'))),
                            'data', 'raw', 'content_refresh_anonymized.csv')

df = pd.read_csv(csv_path)

total_rows = len(df)
unique_clients = df['client_id'].nunique()
declining = (df['trend_direction'] == 'down').sum()
declining_pct = declining / total_rows * 100
word_count_missing = df['word_count'].isna().sum()
word_count_missing_pct = word_count_missing / total_rows * 100

print(f"Total content items : {total_rows:,}")
print(f"Unique clients       : {unique_clients}")
print(f"Declining pages      : {declining:,} ({declining_pct:.1f}% of inventory)")
print(f"Missing word_count   : {word_count_missing:,} ({word_count_missing_pct:.1f}% of rows)")

Total content items : 30,000
Unique clients       : 32
Declining pages      : 16,262 (54.2% of inventory)
Missing word_count   : 7,699 (25.7% of rows)


## 2. The question: decision, action, cost of a wrong call

**The decision:** Which pages in the content inventory should be flagged as high-confidence refresh candidates, justified by observable evidence, so that a human reviewer can accept or reject each recommendation efficiently?

**The action:** A content editor receives each flagged page alongside a plain-language reason code (for example, *"stale, high-visibility page"* or *"page-one decay risk"*). The editor reviews that single page once — a task that should take no more than five minutes — and makes a binary call: schedule it for a full rewrite, or mark it as "no action needed this cycle." The reason code is the primary tool driving that call.

**The cost of a wrong call:** Both error types carry real cost, but asymmetrically. A false positive (flagging a page that didn’t need attention) wastes a review slot and, if it happens repeatedly, erodes the editor’s trust in the queue until they stop using it. A false negative (missing a page that genuinely needed attention) allows a high-visibility page to continue losing impressions and position undetected — visibility that is slow and expensive to recover. The model must therefore hit a minimum recall floor on pages above a defined impression threshold, while keeping precision high enough that editors do not reject the queue as noise. Optimising for **Precision@K** with a **volume-weighted recall constraint** is the right design target.

In [2]:
# Supporting numbers for Section 2 — impression threshold feasibility
# Shows how many declining pages clear the minimum-volume bar (>= 500 impressions_90d)

high_vis_declining = df[
    (df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)
]

print(f"Declining pages (all)                  : {declining:,}")
print(f"Declining + >=500 impressions_90d      : {len(high_vis_declining):,} "
      f"({len(high_vis_declining)/declining*100:.1f}% of declining)")
print()
print("These are the pages where a false negative costs the most —")
print("the recall floor must cover them first.")

Declining pages (all)                  : 16,262
Declining + >=500 impressions_90d      : 9,961 (61.3% of declining)

These are the pages where a false negative costs the most —
the recall floor must cover them first.


## 3. Quick look at the data (2–3 real numbers)

All three numbers below were computed from the starter dataset (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32 clients).

**Number 1 — Scale of the problem:** 16,262 pages (54.2% of the inventory) currently show a declining trend direction. This immediately establishes that the problem cannot be solved by manual review; a ranked, scored queue is operationally necessary.

**Number 2 — The metadata gap:** 7,699 rows (25.7%) are missing `word_count`. The core performance signals — `impressions_90d`, `sessions_90d`, `ctr`, and `avg_position` — are 100% populated, which means the ranking can be built on solid ground, but a blanket rule like *"flag thin pages"* would silently fail on over a quarter of the inventory. A model that encodes missingness explicitly handles this correctly.

**Number 3 — High-stakes subset:** Of the 16,262 declining pages, 5,573 (34.3%) also clear the ≥500 impressions_90d threshold. These are not low-traffic noise; they are visible, actively declining pages where inaction is costly. They define the recall floor the final model must respect.

In [3]:
# Full EDA summary — all three numbers in one block
print("=== Starter Dataset Overview ===")
print(f"Rows                     : {total_rows:,}")
print(f"Unique clients           : {unique_clients}")
print()
print("--- Number 1: Scale of declining inventory ---")
td = df['trend_direction'].value_counts()
for direction, count in td.items():
    print(f"  {direction:<10}: {count:>6,}  ({count/total_rows*100:.1f}%)")
print()
print("--- Number 2: Core signal completeness ---")
for col in ['impressions_90d', 'sessions_90d', 'ctr', 'avg_position', 'word_count']:
    missing = df[col].isna().sum()
    print(f"  {col:<22}: {missing:>5,} missing ({missing/total_rows*100:.1f}%)")
print()
print("--- Number 3: High-stakes declining subset ---")
print(f"  Declining + >=500 impr : {len(high_vis_declining):,} "
      f"({len(high_vis_declining)/declining*100:.1f}% of declining pages)")

=== Starter Dataset Overview ===
Rows                     : 30,000
Unique clients           : 32

--- Number 1: Scale of declining inventory ---
  down      : 16,262  (54.2%)
  stable    :  5,962  (19.9%)
  up        :  4,388  (14.6%)
  new       :  2,236  (7.5%)
  flat      :  1,152  (3.8%)

--- Number 2: Core signal completeness ---
  impressions_90d       :     0 missing (0.0%)
  sessions_90d          :     0 missing (0.0%)
  ctr                   :     0 missing (0.0%)
  avg_position          :     0 missing (0.0%)
  word_count            : 7,699 missing (25.7%)

--- Number 3: High-stakes declining subset ---
  Declining + >=500 impr : 9,961 (61.3% of declining pages)


## 4. Careful words: what I can and can’t claim

**What this work can claim:**

- *Observational:* "We observed that pages with declining trend signals and high impression volume are disproportionately represented in the highest-scoring refresh candidates."
- *Directional:* "The model’s scoring suggests that pages in the top decile of the refresh score share specific structural characteristics — position tier, content age, and impression volume — that distinguish them from lower-ranked pages."
- *Decision-support:* "This ranked queue helps a content editor decide which pages to review first within a given sprint, given finite editorial capacity."

**What this work cannot claim:**

- It cannot prove that a content refresh *caused* a recovery in traffic or rankings. Establishing causality would require a controlled experiment or a valid causal design, neither of which this observational dataset supports.
- It cannot claim to have identified a Google ranking algorithm factor. The signals used are observable search and content metrics; they are correlates, not confirmed drivers.
- It cannot claim the model is correct for any individual page. A high confidence score is a prioritisation signal, not a guarantee. Every flagged page still requires a human review decision.
- It cannot use `trend_direction` or `trend_pct` as model features. These columns are the source of the training label (`is_declining_label`), making them leakage risks that must be excluded.

In [4]:
# Leakage guard: confirm trend_direction and trend_pct will NOT be used as features
# These columns are the direct source of is_declining_label — using them would be circular.

label_source_cols = ['trend_direction', 'trend_pct']
for col in label_source_cols:
    if col in df.columns:
        print(f"EXCLUDED from features (label source): '{col}' — {df[col].nunique()} unique values")

print()
print("Safe observable features present (sample):")
safe_cols = ['impressions_90d', 'sessions_90d', 'ctr', 'avg_position',
             'content_age_days', 'word_count', 'engagement_rate', 'scroll_rate']
for col in safe_cols:
    if col in df.columns:
        print(f"  {col:<25}: min={df[col].min():.2f}, max={df[col].max():.2f}, "
              f"missing={df[col].isna().sum()}")

EXCLUDED from features (label source): 'trend_direction' — 5 unique values
EXCLUDED from features (label source): 'trend_pct' — 2712 unique values

Safe observable features present (sample):
  impressions_90d          : min=1.00, max=517715.00, missing=0
  sessions_90d             : min=1.00, max=4345.00, missing=0
  ctr                      : min=0.00, max=100.00, missing=0
  avg_position             : min=0.00, max=245.00, missing=0
  content_age_days         : min=90.00, max=564.00, missing=0
  word_count               : min=8.00, max=9546.00, missing=7699
  engagement_rate          : min=0.00, max=100.00, missing=0
  scroll_rate              : min=0.00, max=300.00, missing=125


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.